Análise de qualidade dos dados de Tuberculose retirados do SINAN - 2014 à 2024

Vamos analisar a qualidade dos dados com base em completude, unicidade, validade e consistência.

In [1]:
import numpy as np

# 1. O DICIONÁRIO DE DADOS COMPLETO (SINAN TUBERCULOSE 2019)
dicionario_sinan_completo = {
    # --- CHAVES E INFORMAÇÕES DE SISTEMA/NOTIFICAÇÃO ---
    'TP_NOT': 'Tipo_Notificacao',
    'ID_AGRAVO': 'Agravo_Codigo',
    'DT_NOTIFIC': 'Data_Notificacao',
    'NU_ANO': 'Ano_Notificacao',
    'SG_UF_NOT': 'UF_Notificacao',
    'ID_MUNICIP': 'Municipio_Notificacao',
    'ID_REGIONA': 'Regional_Notificacao',
    'TPUNINOT': 'Tipo_Unidade_Notificacao',
    'NDUPLIC_N': 'Indicador_Duplicidade_Sistema',
    'IN_VINCULA': 'Indicador_Vinculo_Sistema',
    'MIGRADO_W': 'Indicador_Migrado_Web',
    
    # --- DATAS DE TRAMITAÇÃO DO SISTEMA (MENOS RELEVANTES CLÍNICAMENTE) ---
    'DT_DIGITA': 'Data_Digitacao_Sistema',
    'DT_TRANSUS': 'Data_Envio_SUS',
    'DT_TRANSDM': 'Data_Envio_Distrito',
    'DT_TRANSSM': 'Data_Envio_Municipio',
    'DT_TRANSRM': 'Data_Envio_Regional',
    'DT_TRANSRS': 'Data_Envio_Estado',
    'DT_TRANSSE': 'Data_Envio_Secretaria_Estadual',
    'CS_FLXRET': 'Indicador_Fluxo_Retorno',
    'FLXRECEBI': 'Indicador_Fluxo_Recebimento',
    
    # --- PERFIL SOCIODEMOGRÁFICO DO PACIENTE ---
    'DT_DIAG': 'Data_Diagnostico',
    'ANO_NASC': 'Ano_Nascimento',
    'NU_IDADE_N': 'Idade_Codigo', 
    'CS_SEXO': 'Sexo',
    'CS_GESTANT': 'Gestante',
    'CS_RACA': 'Raca_Cor',
    'CS_ESCOL_N': 'Escolaridade',
    'ID_OCUPA_N': 'Ocupacao',
    'SG_UF': 'UF_Residencia',
    'ID_MN_RESI': 'Municipio_Residencia',
    'ID_RG_RESI': 'Regional_Residencia',
    'ID_PAIS': 'Pais_Residencia',
    
    # --- POPULAÇÕES ESPECÍFICAS ---
    'POP_LIBER': 'Populacao_Privada_Liberdade',
    'POP_RUA': 'Populacao_Situacao_Rua',
    'POP_SAUDE': 'Profissional_Saude',
    'POP_IMIG': 'Imigrante',
    'BENEF_GOV': 'Beneficiario_Programa_Governo',
    'INSTITUCIO': 'Paciente_Institucionalizado',
    
    # --- CLÍNICA E COMORBIDADES (AGRAVOS) ---
    'TRATAMENTO': 'Tipo_Entrada', # Caso Novo, Recidiva, Reingresso
    'FORMA': 'Forma_Clinica',
    'EXTRAPU1_N': 'Local_Extrapulmonar_1',
    'EXTRAPU2_N': 'Local_Extrapulmonar_2',
    'AGRAVAIDS': 'Agravo_AIDS',
    'AGRAVALCOO': 'Agravo_Alcoolismo',
    'AGRAVDIABE': 'Agravo_Diabetes',
    'AGRAVDOENC': 'Agravo_Doenca_Mental',
    'AGRAVTABAC': 'Agravo_Tabagismo',
    'AGRAVDROGA': 'Agravo_Drogas_Ilicitas',
    'AGRAVOUTRA': 'Agravo_Outras_Doencas',
    'HIV': 'Sorologia_HIV',
    'ANT_RETRO': 'Uso_Antiretroviral',
    'DOENCA_TRA': 'Relacionado_Doenca_Trabalho',
    
    # --- EXAMES DE DIAGNÓSTICO E ACOMPANHAMENTO ---
    'RAIOX_TORA': 'RaioX_Torax',
    'TESTE_TUBE': 'Teste_Tuberculinico',
    'TEST_MOLEC': 'Teste_Rapido_Molecular',
    'TEST_SENSI': 'Teste_Sensibilidade',
    'HISTOPATOL': 'Exame_Histopatologico',
    'BACILOSC_E': 'Baciloscopia_Escarro_Diagnostico',
    'CULTURA_ES': 'Cultura_Escarro',
    'CULTURA_OU': 'Cultura_Outro_Material',
    'BACILOS_E2': 'Baciloscopia_Escarro_2_Diagnostico',
    'BACILOSC_O': 'Baciloscopia_Outros_Materiais',
    
    # --- ACOMPANHAMENTO MENSAL (BACILOSCOPIA) ---
    'BACILOSC_1': 'Baciloscopia_1_Mes',
    'BACILOSC_2': 'Baciloscopia_2_Mes',
    'BACILOSC_3': 'Baciloscopia_3_Mes',
    'BACILOSC_4': 'Baciloscopia_4_Mes',
    'BACILOSC_5': 'Baciloscopia_5_Mes',
    'BACILOSC_6': 'Baciloscopia_6_Mes',
    'BAC_APOS_6': 'Baciloscopia_Apos_6_Meses',
    
    # --- TRATAMENTO ---
    'DT_INIC_TR': 'Data_Inicio_Tratamento',
    'RIFAMPICIN': 'Droga_Rifampicina',
    'ISONIAZIDA': 'Droga_Isoniazida',
    'ETAMBUTOL': 'Droga_Etambutol',
    'ESTREPTOMI': 'Droga_Estreptomicina',
    'PIRAZINAMI': 'Droga_Pirazinamida',
    'ETIONAMIDA': 'Droga_Etionamida',
    'OUTRAS': 'Drogas_Outras',
    'DT_MUDANCA': 'Data_Mudanca_Esquema',
    'TRAT_SUPER': 'Tratamento_Supervisionado_Realizado',
    
    # --- TRANSFERÊNCIAS DURANTE O TRATAMENTO ---
    'SG_UF_AT': 'UF_Atual_Tratamento',
    'ID_MUNIC_A': 'Municipio_Atual_Tratamento',
    'DT_NOTI_AT': 'Data_Atualizacao_Notificacao',
    'TRANSF': 'Transferencia_Confirmada',
    'UF_TRANSF': 'UF_Transferencia',
    'MUN_TRANSF': 'Municipio_Transferencia',
    'SG_UF_2': 'UF_2_Transferencia',
    'ID_MUNIC_2': 'Municipio_2_Transferencia',
    'TRATSUP_AT': 'Tratamento_Supervisionado_Atual',
    
    # --- CONTATOS ---
    'NU_CONTATO': 'Numero_Contatos_Identificados',
    'NU_COMU_EX': 'Numero_Contatos_Examinados',
    
    # --- DESFECHO / ENCERRAMENTO ---
    'SITUA_9_M': 'Situacao_9_Meses',
    'SITUA_12_M': 'Situacao_12_Meses',
    'SITUA_ENCE': 'Situacao_Encerramento_Final',
    'DT_ENCERRA': 'Data_Encerramento'
}


In [2]:
import glob
import pandas as pd

arquivos_csv = glob.glob('*.csv')
casos_por_ano = {}

print("Contando notificações por arquivo. Isso será rápido...\n")

for arquivo in arquivos_csv:
    try:
        # DICA DE SÊNIOR: usecols=[0] lê apenas a primeira coluna do arquivo.
        # Isso reduz o uso de memória em 99% e conta as linhas em segundos.
        df_temp = pd.read_csv(arquivo, sep=',', encoding='latin1', on_bad_lines='warn', usecols=[0])
        
        # Guardando o total de linhas (casos)
        total_linhas = len(df_temp)
        casos_por_ano[arquivo] = total_linhas
        
        print(f"✔️ {arquivo}: {total_linhas} casos")
        
    except Exception as e:
        print(f"⚠️ Erro ao ler o arquivo {arquivo}: {e}")

# Transformando o resultado em uma tabela ordenada
df_volumetria = pd.DataFrame(list(casos_por_ano.items()), columns=['Arquivo', 'Total de Casos'])
df_volumetria = df_volumetria.sort_values(by='Total de Casos', ascending=False).reset_index(drop=True)

# Formatando para facilitar a leitura com separador de milhar
df_volumetria['Total de Casos'] = df_volumetria['Total de Casos'].apply(lambda x: f"{x:,}".replace(',', '.'))

print("\n--- RANKING DE VOLUMETRIA (BRASIL) ---")
display(df_volumetria)

# Identificando o pico
arquivo_pico = df_volumetria.iloc[0]['Arquivo']
casos_pico = df_volumetria.iloc[0]['Total de Casos']

print(f"\n🚨 O pico histórico ocorreu no arquivo '{arquivo_pico}', com {casos_pico} notificações registradas.")

Contando notificações por arquivo. Isso será rápido...

✔️ TUBEBR15.csv: 85462 casos
✔️ TUBEBR22.csv: 103330 casos
✔️ TUBEBR18.csv: 94735 casos
✔️ TUBEBR14.csv: 85216 casos
✔️ TUBEBR19.csv: 95849 casos
✔️ TUBEBR23.csv: 109854 casos
✔️ TUBEBR20.csv: 85962 casos
✔️ TUBEBR24.csv: 112988 casos
✔️ TUBEBR21.csv: 91310 casos
✔️ TUBEBR16.csv: 86210 casos
✔️ TUBEBR17.csv: 90295 casos

--- RANKING DE VOLUMETRIA (BRASIL) ---


,Arquivo,Total de Casos
0,TUBEBR24.csv,112.988
1,TUBEBR23.csv,109.854
2,TUBEBR22.csv,103.330
3,TUBEBR19.csv,95.849
4,TUBEBR18.csv,94.735
5,TUBEBR21.csv,91.310
6,TUBEBR17.csv,90.295
7,TUBEBR16.csv,86.210
8,TUBEBR20.csv,85.962
9,TUBEBR15.csv,85.462



🚨 O pico histórico ocorreu no arquivo 'TUBEBR24.csv', com 112.988 notificações registradas.


In [3]:
# Carregamento do arquivo de 2024 e aplicação do dicionário
df_2024 = pd.read_csv('TUBEBR24.csv', sep=',', encoding='latin1', on_bad_lines='warn', dtype=str)
df_2024.columns = df_2024.columns.str.strip()
df_2024.rename(columns=dicionario_sinan_completo, inplace=True, errors='ignore')

# 1. TRATAMENTO REFORÇADO DE FALSOS NULOS
# O regex '^\s*$' pega células que estão vazias ou contêm apenas espaços
df_2024.replace(r'^\s*$', np.nan, regex=True, inplace=True)

# Tratando também possíveis strings de nulos que o Pandas não capturou nativamente 
# (como 'NaN', 'nan', 'NaM', 'None' em formato de texto)
df_2024.replace(['NaN', 'nan', 'NaM', 'None', ''], np.nan, inplace=True)

# 2. CÁLCULO DAS MÉTRICAS DE COMPLETUDE
total_registros_2024 = len(df_2024)
nulos_absolutos_2024 = df_2024.isnull().sum()
percentual_nulos_2024 = (nulos_absolutos_2024 / total_registros_2024) * 100

# 3. CRIAÇÃO DA TABELA
tabela_completude_2024 = pd.DataFrame({
    'Variável': df_2024.columns,
    'Total de Nulos': nulos_absolutos_2024.values,
    'Dados Ausentes (%)': percentual_nulos_2024.values
})

# Ordenando as colunas mais críticas (com mais nulos) para o topo
tabela_completude_2024.sort_values(by='Dados Ausentes (%)', ascending=False, inplace=True)
tabela_completude_2024.reset_index(drop=True, inplace=True)

# 4. VISUALIZAÇÃO SÊNIOR (TABELA ESTILIZADA)
# Aplica um gradiente de cor na coluna de porcentagem: 
# Vermelho para muitos nulos (ruim), Verde para poucos nulos (bom)
tabela_visual_2024 = tabela_completude_2024.style.format({
    'Total de Nulos': '{:,}',           # Formata com separador de milhar
    'Dados Ausentes (%)': '{:.2f}%'     # Formata com duas casas decimais e símbolo de %
}).background_gradient(
    cmap='Reds', # Usa tons de vermelho
    subset=['Dados Ausentes (%)']
)

# Exibe a tabela no Jupyter
display(tabela_visual_2024)

/tmp/ipykernel_10035/3782869052.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_2024.replace(r'^\s*$', np.nan, regex=True, inplace=True)


,Variável,Total de Nulos,Dados Ausentes (%)
0,Indicador_Fluxo_Recebimento,"112,988",100.00%
1,Indicador_Fluxo_Retorno,"112,988",100.00%
2,Data_Envio_Regional,"112,988",100.00%
3,Indicador_Migrado_Web,"112,988",100.00%
4,Data_Mudanca_Esquema,"112,988",100.00%
5,Situacao_12_Meses,"112,986",100.00%
6,Droga_Pirazinamida,"112,985",100.00%
7,Droga_Etionamida,"112,985",100.00%
8,Drogas_Outras,"112,985",100.00%
9,Droga_Etambutol,"112,985",100.00%


In [4]:
# 1. MAPEAMENTO E CONFIGURAÇÃO
arquivos_csv = glob.glob('*.csv') # Pega todos os CSVs na pasta onde está o notebook
codigo_ilheus_itabuna = ['291360', '291480']
dfs_decada = []

print(f"Encontrados {len(arquivos_csv)} arquivos CSV. Iniciando extração e filtragem...\n")

# 2. CARREGAMENTO COM FILTRAGEM INTELIGENTE (MEMORY-EFFICIENT)
for arquivo in arquivos_csv:
    try:
        # Lê o arquivo como texto para evitar problemas de tipagem
        df_temp = pd.read_csv(arquivo, sep=',', encoding='latin1', on_bad_lines='warn', dtype=str)
        
        # Remove espaços ocultos no nome das colunas
        df_temp.columns = df_temp.columns.str.strip()
        
        # Aplica o nosso dicionário validado
        df_temp.rename(columns=dicionario_sinan_completo, inplace=True, errors='ignore')
        
        # Filtra IMEDIATAMENTE as cidades desejadas para economizar memória
        if 'Municipio_Notificacao' in df_temp.columns:
            # Limpa espaços em branco que o SINAN possa ter deixado no código
            df_temp['Municipio_Notificacao'] = df_temp['Municipio_Notificacao'].str.strip()
            
            df_cidades = df_temp[df_temp['Municipio_Notificacao'].isin(codigo_ilheus_itabuna)].copy()
            
            if not df_cidades.empty:
                dfs_decada.append(df_cidades)
                print(f"✔️ {arquivo}: {len(df_cidades)} casos encontrados em Ilhéus/Itabuna.")
            else:
                print(f"⚠️ {arquivo}: Nenhum caso encontrado para essas cidades.")
                
    except Exception as e:
        print(f"Erro ao processar o arquivo {arquivo}: {e}")

# 3. CONSOLIDAÇÃO DA DÉCADA
df_regiao_decada = pd.concat(dfs_decada, ignore_index=True)

print(f"\nExtração concluída! Total da década (2014-2024): {len(df_regiao_decada)} notificações.\n")

# 4. TRATAMENTO DE NULOS E CÁLCULO DE COMPLETUDE
# Limpando espaços, strings vazias e 'NaN' em texto
df_regiao_decada = df_regiao_decada.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
df_regiao_decada.replace(r'^\s*$', np.nan, regex=True, inplace=True)
df_regiao_decada.replace(['NaN', 'nan', 'NaM', 'None', ''], np.nan, inplace=True)

total_registros_decada = len(df_regiao_decada)
nulos_decada = df_regiao_decada.isnull().sum()
perc_nulos_decada = (nulos_decada / total_registros_decada) * 100

# 5. TABELA VISUAL
tabela_completude_decada = pd.DataFrame({
    'Variável': df_regiao_decada.columns,
    'Total de Nulos (10 Anos)': nulos_decada.values,
    'Dados Ausentes (%)': perc_nulos_decada.values
})

tabela_completude_decada.sort_values(by='Dados Ausentes (%)', ascending=False, inplace=True)
tabela_completude_decada.reset_index(drop=True, inplace=True)

tabela_visual_decada = tabela_completude_decada.style.format({
    'Total de Nulos (10 Anos)': '{:,}',           
    'Dados Ausentes (%)': '{:.2f}%'     
}).background_gradient(
    cmap='Reds', 
    subset=['Dados Ausentes (%)']
)

# Exibe a tabela consolidada
display(tabela_visual_decada)

Encontrados 11 arquivos CSV. Iniciando extração e filtragem...

✔️ TUBEBR15.csv: 260 casos encontrados em Ilhéus/Itabuna.
✔️ TUBEBR22.csv: 356 casos encontrados em Ilhéus/Itabuna.
✔️ TUBEBR18.csv: 300 casos encontrados em Ilhéus/Itabuna.
✔️ TUBEBR14.csv: 270 casos encontrados em Ilhéus/Itabuna.
✔️ TUBEBR19.csv: 348 casos encontrados em Ilhéus/Itabuna.
✔️ TUBEBR23.csv: 283 casos encontrados em Ilhéus/Itabuna.
✔️ TUBEBR20.csv: 290 casos encontrados em Ilhéus/Itabuna.
✔️ TUBEBR24.csv: 344 casos encontrados em Ilhéus/Itabuna.
✔️ TUBEBR21.csv: 345 casos encontrados em Ilhéus/Itabuna.
✔️ TUBEBR16.csv: 314 casos encontrados em Ilhéus/Itabuna.
✔️ TUBEBR17.csv: 321 casos encontrados em Ilhéus/Itabuna.

Extração concluída! Total da década (2014-2024): 3431 notificações.



/tmp/ipykernel_10035/3083454630.py:44: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_regiao_decada.replace(r'^\s*$', np.nan, regex=True, inplace=True)


,Variável,Total de Nulos (10 Anos),Dados Ausentes (%)
0,Indicador_Fluxo_Recebimento,"3,431",100.00%
1,Data_Envio_Distrito,"3,431",100.00%
2,Data_Envio_SUS,"3,431",100.00%
3,Data_Envio_Regional,"3,431",100.00%
4,Indicador_Fluxo_Retorno,"3,431",100.00%
5,Data_Envio_Secretaria_Estadual,"3,430",99.97%
6,EXTRAPUL_O,"3,423",99.77%
7,AGRAVOUTDE,"3,347",97.55%
8,Municipio_Transferencia,"3,271",95.34%
9,UF_Transferencia,"3,271",95.34%


In [5]:
# 1. LISTA ATUALIZADA DE EXCEÇÕES (Nulos Informacionais)
excecoes_conhecidas = [
    # Acompanhamento Mensal
    'Baciloscopia_1_Mes', 'Baciloscopia_2_Mes', 'Baciloscopia_3_Mes',
    'Baciloscopia_4_Mes', 'Baciloscopia_5_Mes', 'Baciloscopia_6_Mes',
    'Baciloscopia_Apos_6_Meses', 'Situacao_9_Meses', 'Situacao_12_Meses',
    
    # Eventos Condicionais e Raros
    'Transferencia_Confirmada', 'UF_Transferencia', 'Municipio_Transferencia',
    'Data_Mudanca_Esquema', 'Local_Extrapulmonar_2', 'EXTRAPUL_O',
    'AGRAVOUTDE', 'OUTRAS_DES', 
    
    # Exames Específicos
    'Baciloscopia_Outros_Materiais', 'Cultura_Outro_Material', 'Baciloscopia_Escarro_2_Diagnostico'
]

# 2. IDENTIFICAÇÃO DAS COLUNAS CRÍTICAS (>= 45%)
colunas_criticas = perc_nulos_decada[perc_nulos_decada >= 45.0].index.tolist()

# 3. EXECUTANDO A LIMPEZA DEFINTIVA
# Variáveis que estão na lista crítica MAS NÃO estão nas exceções
colunas_para_deletar = [col for col in colunas_criticas if col not in excecoes_conhecidas]

# Deletando do dataframe da década
df_limpo = df_regiao_decada.drop(columns=colunas_para_deletar).copy()

print(f" Faxina Concluída!")
print(f" Variáveis descartadas: {len(colunas_para_deletar)}")
print(f" Variáveis mantidas no dataset final: {len(df_limpo.columns)}")

 Faxina Concluída!
 Variáveis descartadas: 23
 Variáveis mantidas no dataset final: 74


In [6]:
# 1. RECUPERANDO AS VARIÁVEIS APROVADAS
colunas_aprovadas = df_limpo.columns.tolist()

# 2. CRIANDO UM RESUMO VISUAL DO CHECKPOINT
# Vamos criar uma tabela mostrando as colunas que ficaram e o motivo
df_resumo_aprovadas = pd.DataFrame({
    'Variáveis Aprovadas (No Dataset Final)': colunas_aprovadas,
    'Dados Ausentes (%)': perc_nulos_decada[colunas_aprovadas].values
})

# Identificando o motivo da manutenção
df_resumo_aprovadas['Motivo da Manutenção'] = df_resumo_aprovadas['Variáveis Aprovadas (No Dataset Final)'].apply(
    lambda x: '🛡️ Exceção Clínica Valiosa' if x in excecoes_conhecidas else '✅ Preenchimento Adequado (<45%)'
)

# Ordenando para facilitar a leitura (das exceções mais nulas para os dados mais preenchidos)
df_resumo_aprovadas.sort_values(by=['Motivo da Manutenção', 'Dados Ausentes (%)'], ascending=[False, False], inplace=True)
df_resumo_aprovadas.reset_index(drop=True, inplace=True)

# 3. EXIBIÇÃO SÊNIOR
print(f"✅ CHECKPOINT DE SUCESSO")
print(f"Total de Variáveis Prontas para a Próxima Fase: {len(colunas_aprovadas)}\n")

tabela_visual_aprovadas = df_resumo_aprovadas.style.format({
    'Dados Ausentes (%)': '{:.2f}%'
}).applymap(
    lambda val: 'color: blue; font-weight: bold' if 'Exceção' in val else 'color: green',
    subset=['Motivo da Manutenção']
)

# Exibe a tabela interativa
display(tabela_visual_aprovadas)

# Opcional: Se você quiser imprimir apenas a lista de texto corrido (para copiar/colar):
print("\n--- Lista em formato de texto ---")
print(colunas_aprovadas)

✅ CHECKPOINT DE SUCESSO
Total de Variáveis Prontas para a Próxima Fase: 74



/tmp/ipykernel_10035/4060290270.py:26: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  }).applymap(


,Variáveis Aprovadas (No Dataset Final),Dados Ausentes (%),Motivo da Manutenção
0,EXTRAPUL_O,99.77%,🛡️ Exceção Clínica Valiosa
1,AGRAVOUTDE,97.55%,🛡️ Exceção Clínica Valiosa
2,UF_Transferencia,95.34%,🛡️ Exceção Clínica Valiosa
3,Municipio_Transferencia,95.34%,🛡️ Exceção Clínica Valiosa
4,Situacao_12_Meses,90.82%,🛡️ Exceção Clínica Valiosa
5,OUTRAS_DES,81.49%,🛡️ Exceção Clínica Valiosa
6,Local_Extrapulmonar_2,66.04%,🛡️ Exceção Clínica Valiosa
7,Baciloscopia_Escarro_2_Diagnostico,66.04%,🛡️ Exceção Clínica Valiosa
8,Baciloscopia_Outros_Materiais,66.04%,🛡️ Exceção Clínica Valiosa
9,Cultura_Outro_Material,66.04%,🛡️ Exceção Clínica Valiosa



--- Lista em formato de texto ---
['Tipo_Notificacao', 'Agravo_Codigo', 'Data_Notificacao', 'Ano_Notificacao', 'UF_Notificacao', 'Municipio_Notificacao', 'Regional_Notificacao', 'Data_Diagnostico', 'Ano_Nascimento', 'Idade_Codigo', 'Sexo', 'Gestante', 'Raca_Cor', 'Escolaridade', 'UF_Residencia', 'Municipio_Residencia', 'Regional_Residencia', 'Pais_Residencia', 'Data_Digitacao_Sistema', 'Tipo_Entrada', 'RaioX_Torax', 'Forma_Clinica', 'Local_Extrapulmonar_1', 'Local_Extrapulmonar_2', 'EXTRAPUL_O', 'Agravo_AIDS', 'Agravo_Alcoolismo', 'Agravo_Diabetes', 'Agravo_Doenca_Mental', 'Agravo_Outras_Doencas', 'AGRAVOUTDE', 'Baciloscopia_Escarro_Diagnostico', 'Baciloscopia_Escarro_2_Diagnostico', 'Baciloscopia_Outros_Materiais', 'Cultura_Escarro', 'Cultura_Outro_Material', 'Sorologia_HIV', 'Exame_Histopatologico', 'Data_Inicio_Tratamento', 'OUTRAS_DES', 'Numero_Contatos_Identificados', 'UF_Atual_Tratamento', 'Municipio_Atual_Tratamento', 'Data_Atualizacao_Notificacao', 'UF_2_Transferencia', 'Munic

In [7]:
import json

# 1. SALVANDO O DATASET LIMPO
# index=False evita que o Pandas crie uma coluna extra inútil com o número da linha
nome_arquivo_dados = 'tuberculose_ilheus_itabuna_limpo_2014_2024.csv'
df_limpo.to_csv(nome_arquivo_dados, index=False, encoding='utf-8')

# 2. SALVANDO O DICIONÁRIO EM JSON
# ensure_ascii=False garante que acentos (se houver) fiquem legíveis no arquivo
# indent=4 deixa o arquivo JSON organizado e fácil de ler caso você o abra no VS Code
nome_arquivo_json = 'dicionario_sinan_tuberculose.json'
with open(nome_arquivo_json, 'w', encoding='utf-8') as f:
    json.dump(dicionario_sinan_completo, f, ensure_ascii=False, indent=4)

print(f"✅ SUCESSO! Arquivos salvos na sua pasta:")
print(f" 📊 Dados: {nome_arquivo_dados}")
print(f" 📖 Dicionário: {nome_arquivo_json}")
print("\nVocê já pode criar um novo notebook para a etapa de Formatação e Tipagem!")

✅ SUCESSO! Arquivos salvos na sua pasta:
 📊 Dados: tuberculose_ilheus_itabuna_limpo_2014_2024.csv
 📖 Dicionário: dicionario_sinan_tuberculose.json

Você já pode criar um novo notebook para a etapa de Formatação e Tipagem!
